# Record garage pick-and-place demos by teleoperation

The task string is set for you each episode (shown in the viewer as **Task**): *pick up the {tool} and place it in the tray*.
Tools are cycled wrench -> screwdriver -> hammer -> pliers -> ... so the dataset stays balanced.
An episode is saved automatically when the tool is in the green tray, the gripper is open and the gripper is lifted above it.

| Key | Action |
|---|---|
| W / S | arm x: toward robot / away |
| A / D | arm y: left / right (base stays) |
| R / F | arm up / down |
| **J / L** | **linear actuator: whole arm left / right** |
| Q / E | yaw |
| arrow keys | pitch / roll |
| SPACE | toggle gripper |
| Z | discard this episode and reshuffle |

Click the MuJoCo window first so it receives the key presses.

In [12]:
import os, shutil
import numpy as np
from PIL import Image
from mujoco_env.y_env_tools import SimpleEnvTools, TOOLS, PROMPT
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

In [13]:
SEED = 0                      # None -> different object layout every run
REPO_NAME = 'garage_pnp'
ROOT = './demo_data_garage'
NUM_DEMO = 5                 # episodes to record in this session
XML = './asset/example_scene_tools.xml'
TOOL_CYCLE = list(TOOLS)      # order of tools; e.g. ['hammer','hammer','wrench'] to bias

In [14]:
create_new = True
if os.path.exists(ROOT):
    ans = input(f"{ROOT} already exists. Delete it and start fresh? (y = delete, n = append to it) ")
    if ans == 'y':
        shutil.rmtree(ROOT)
    else:
        create_new = False

if create_new:
    img = lambda n: {"dtype": "image", "shape": (256, 256, 3), "names": ["height", "width", n]}
    dataset = LeRobotDataset.create(
        repo_id=REPO_NAME, root=ROOT, robot_type="omy_slider", fps=20,
        features={
            "observation.image": img("channels"),
            "observation.wrist_image": img("channel"),
            "observation.state": {"dtype": "float32", "shape": (8,), "names": ["state"]},   # 6 joints, slide, gripper
            "action": {"dtype": "float32", "shape": (8,), "names": ["action"]},             # 6 joints, slide, gripper
            "obj_init": {"dtype": "float32", "shape": (16,), "names": ["obj_init"]},        # 4 tools x (x,y,z,yaw)
        },
        image_writer_threads=10, image_writer_processes=5)
else:
    print("Appending to the existing dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)

./demo_data_garage already exists. Delete it and start fresh? (y = delete, n = append to it)  y


In [5]:
PnPEnv = SimpleEnvTools(XML, action_type='eef_pose', seed=SEED)
PnPEnv.slide_mode = 'manual'     # J / L drive the linear actuator


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[39] n_qvel:[35] n_qacc:[35] n_ctrl:[11]
 integrator:[IMPLICITFAST]

n_body:[24]
 [0/24] [world] mass:[0.00]kg
 [1/24] [front_object_table] mass:[1.00]kg
 [2/24] [camera] mass:[0.00]kg
 [3/24] [camera2] mass:[0.00]kg
 [4/24] [camera3] mass:[0.00]kg
 [5/24] [slider_carriage] mass:[12.00]kg
 [6/24] [link1] mass:[2.06]kg
 [7/24] [link2] mass:[3.68]kg
 [8/24] [link3] mass:[2.39]kg
 [9/24] [link4] mass:[1.40]kg
 [10/24] [link5] mass:[1.40]kg
 [11/24] [link6] mass:[0.65]kg
 [12/24] [camera_center] mass:[0.00]kg
 [13/24] [tcp_link] mass:[0.32]kg
 [14/24] [rh_p12_rn_r1] mass:[0.07]kg
 [15/24] [rh_p12_rn_r2] mass:[0.02]kg
 [16/24] [rh_p12_rn_l1] mass:[0.07]kg
 [17/24] [rh_p12_rn_l2] mass:[0.02]kg
 [18/24] [body_tray_src] mass:[4.34]kg
 [19/24] [body_tray_dst] mass:[4.34]kg
 [20/24] [body_obj_wrench] mass:[0.09]kg
 [21/24] [body_obj_screwdriver] mass:[0.07]kg
 [22/24] [body_

In [6]:
def start_episode(i):
    """reshuffle the scene (0-2 tools pre-placed in the green tray as distractors) and set the task"""
    tool = TOOL_CYCLE[i % len(TOOL_CYCLE)]
    PnPEnv.reset(n_pre_in_dst=int(np.random.choice([0, 0, 1, 2])), target=tool)   # target never starts in the green tray
    PnPEnv.set_instruction(PROMPT.format(tool))
    print(f"[episode {i}] {PnPEnv.instruction}")
    return tool

MIN_FRAMES = 40      # an episode shorter than 2 s is never accepted as 'success'

def resize(a):
    return np.array(Image.fromarray(a).resize((256, 256)))

### Teleoperate and record  (stop early by closing the viewer window)

In [7]:
episode_id = 0
tool = start_episode(episode_id)
record_flag = False                      # recording starts at your first key press
n_frames = 0

while PnPEnv.env.is_viewer_alive() and episode_id < NUM_DEMO:
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # ---- success -> save and move on to the next tool
        if record_flag and n_frames >= MIN_FRAMES and PnPEnv.check_success(tool):
            dataset.save_episode()
            episode_id += 1
            print(f"saved. total episodes this session: {episode_id}")
            if episode_id >= NUM_DEMO:
                break
            tool = start_episode(episode_id)
            record_flag = False
            n_frames = 0
            continue

        action, reset = PnPEnv.teleop_robot()

        # ---- 'Z' key: throw this attempt away
        if reset:
            dataset.clear_episode_buffer()
            tool = start_episode(episode_id)
            record_flag = False
            n_frames = 0
            continue

        if not record_flag and (np.abs(action).sum() > 1e-6 or PnPEnv.slide_delta != 0):
            record_flag = True
            print("Start recording")

        rgb, wrist = PnPEnv.grab_image()               # images BEFORE the step, like the original notebook
        state = PnPEnv.step(action)                    # 8-D state: 6 joints, slide, gripper
        if record_flag:
            dataset.add_frame({
                "observation.image": resize(rgb),
                "observation.wrist_image": resize(wrist),
                "observation.state": state,
                "action": PnPEnv.get_action_vec(),     # 8-D: 6 joint targets, slide target, gripper
                "obj_init": PnPEnv.obj_init_pose,
            }, task=PnPEnv.instruction)
            n_frames += 1
        PnPEnv.render(teleop=True, idx=episode_id)

DONE INITIALIZATION
[episode 0] pick up the wrench and place it in the tray
Start recording


Map:   0%|          | 0/363 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

saved. total episodes this session: 1
DONE INITIALIZATION
[episode 1] pick up the screwdriver and place it in the tray
Start recording


Map:   0%|          | 0/389 [00:00<?, ? examples/s]

...

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

saved. total episodes this session: 4
DONE INITIALIZATION
[episode 4] pick up the wrench and place it in the tray
Start recording


Map:   0%|          | 0/378 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

saved. total episodes this session: 5


In [8]:
PnPEnv.env.close_viewer()

### Clean up the temporary image folder (run once after you finish recording)

In [11]:
shutil.rmtree(dataset.root / 'images', ignore_errors=True)
print("episodes in dataset:", dataset.num_episodes if hasattr(dataset, 'num_episodes') else dataset.meta.total_episodes)

episodes in dataset: 5
